In [1]:
import pandas as pd
from pathlib import Path
from janitor import clean_names

In [2]:
input_path = Path("/Users/mvilla/Library/CloudStorage/OneDrive-STIM-TheSeattleTimes/Repos_Data/spd_crime_data_2025-10-10_DATA/SPD_Crime_Data")
output_path = Path("../output")

## Crime data

Source: [SPD Crime plot_data: 2008-Present](https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5/)

In [3]:
crime = pd.read_csv(input_path / "SPD_Crime_Data_2018-01-01_2026-08-31.csv", 
                     dtype=str, 
                     ).clean_names()

# Convert date columns to appropriate format
date_cols = ["report_datetime", "offense_date"]
crime[date_cols] = crime[date_cols].apply(lambda x: pd.to_datetime(x, format='%Y %b %d %I:%M:%S %p'))

# Delete any offenses committed outside the desired period

mask = crime["offense_date"].gt("2026-01-01")
print(len(crime))
crime = crime.loc[mask]
print(len(crime))

# Create month column
crime["month"] = crime["offense_date"].dt.month

# Delete any duplicated offense IDs
crime = crime.drop_duplicates(subset=["offense_id"])
print(len(crime))

# Conver float columns to appropriate format
coords = ["longitude", "latitude"]
crime[coords] = crime[coords].apply(lambda x: pd.to_numeric(x, errors="coerce"))


726681
49380
49373


In [4]:
import geopandas as gpd

crime_gdf = gpd.GeoDataFrame(
    crime,
    geometry=gpd.points_from_xy(
        crime.longitude,
        crime.latitude
    ),
    crs="EPSG:4326"
)

In [5]:
from shapely.geometry import Polygon
import geopandas as gpd

# Coordinates are (longitude, latitude), going around the perimeter
coords = [
    (-122.3555552, 47.6977394), 
    (-122.3556209, 47.7052777),   
    (-122.339164, 47.705123),   
    (-122.3393611, 47.697651),  
]

study_area = gpd.GeoDataFrame(
    {"area": ["N 95th–N 105th / Greenwood–College"]},
    geometry=[Polygon(coords)],
    crs="EPSG:4326"
)

study_area.explore()

In [6]:
crime_in_area = gpd.sjoin(
    crime_gdf,
    study_area,
    how="inner",
    predicate="within"
)

In [8]:
table = (
    crime_in_area
     .pivot_table(index=["offense_category", "offense_sub_category"],
                  columns="month",
                  values="report_number",
                  aggfunc="nunique"
                 )
)

table.to_csv("~/Downloads/aurora_square_crimes.csv")
! open ~/Downloads/aurora_square_crimes.csv
table

/Users/mvilla/.pyenv/versions/3.12.11/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=66883) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


month                                                                  1  \
offense_category offense_sub_category                                      
ALL OTHER        999                                                 6.0   
                 ALL OTHER                                           2.0   
                 ANIMAL CRUELTY                                      NaN   
                 ASSAULT OFFENSES                                    3.0   
                 DUI                                                 1.0   
                 EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD C...  3.0   
                 NARCOTIC VIOLATIONS (INCLUDES DRUG EQUIP.)          2.0   
                 PROPERTY OFFENSES (INCLUDES STOLEN, DESTRUCTION)    7.0   
                 PROSTITUTION OFFENSES                               NaN   
                 SEX OFFENSES                                        NaN   
                 TRESPASS                                            NaN   
                 WEAPON LAW VIOLATION                                4.0   
PROPERTY CRIME   ARSON                                               2.0   
                 BURGLARY                                            2.0   
                 LARCENY-THEFT                                       6.0   
                 MOTOR VEHICLE THEFT                                 7.0   
VIOLENT CRIME    AGGRAVATED ASSAULT                                  2.0   
                 ROBBERY                                             1.0   

month                                                                   2  \
offense_category offense_sub_category                                       
ALL OTHER        999                                                  4.0   
                 ALL OTHER                                            NaN   
                 ANIMAL CRUELTY                                       NaN   
                 ASSAULT OFFENSES                                     5.0   
                 DUI                                                  NaN   
                 EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD C...   3.0   
                 NARCOTIC VIOLATIONS (INCLUDES DRUG EQUIP.)           1.0   
                 PROPERTY OFFENSES (INCLUDES STOLEN, DESTRUCTION)     6.0   
                 PROSTITUTION OFFENSES                                NaN   
                 SEX OFFENSES                                         1.0   
                 TRESPASS                                             1.0   
                 WEAPON LAW VIOLATION                                 1.0   
PROPERTY CRIME   ARSON                                                NaN   
                 BURGLARY                                             3.0   
                 LARCENY-THEFT                                       11.0   
                 MOTOR VEHICLE THEFT                                  3.0   
VIOLENT CRIME    AGGRAVATED ASSAULT                                   3.0   
                 ROBBERY                                              3.0   

month                                                                  3    4  \
offense_category offense_sub_category                                           
ALL OTHER        999                                                 8.0  2.0   
                 ALL OTHER                                           1.0  2.0   
                 ANIMAL CRUELTY                                      NaN  NaN   
                 ASSAULT OFFENSES                                    8.0  1.0   
                 DUI                                                 3.0  1.0   
                 EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD C...  5.0  1.0   
                 NARCOTIC VIOLATIONS (INCLUDES DRUG EQUIP.)          NaN  NaN   
                 PROPERTY OFFENSES (INCLUDES STOLEN, DESTRUCTION)    1.0  4.0   
                 PROSTITUTION OFFENSES                               NaN  1.0   
                 SEX OFFENSES                  